# AST Platt recipient verification ? no training
Run on Account B in the existing configured T4 environment. This reads canonical operating audio for at most 12 fixed recordings, including the four closest to the decision boundary. It never fits, updates or promotes anything. Reports are new files only. A passing sample still needs teammate integration review. CPU FP32 is not verified.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project') # Your Account B project copy containing existing data
VERIFY = ROOT / 'experiments/r3_n1/ast_platt_verification'
CANDIDATE = ROOT / 'runs/r3/r3_n1_ast_platt_candidates/20261114T150254Z-046f980d'


In [ ]:
import hashlib, json, os, subprocess, sys, uuid
def sha(p):
    with Path(p).open('rb') as f: return hashlib.file_digest(f,'sha256').hexdigest()
assert sha(VERIFY/'source_manifest.json') == '8464e6bc33dd76a0b8701532a14200e383c8f76dd753ddc33a8b7874aeecd265', 'Wrong verification source manifest'
for name, expected in json.loads((VERIFY/'source_manifest.json').read_text()).items():
    assert sha(VERIFY/name) == expected, 'Source mismatch: '+name
assert (CANDIDATE/'manifest.json').is_file(), 'Upload the complete Platt candidate folder'
import torch, transformers, soundfile, scipy, pyarrow
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'torch:', torch.__version__, 'transformers:', transformers.__version__)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
assert transformers.__version__ == '5.22.1', 'Candidate AST was packaged with transformers 5.22.1. Report this environment mismatch; do not silently change versions.'
ENV = dict(os.environ, PYTHONPATH=str(VERIFY/'source/src'), PYTHONDONTWRITEBYTECODE='1')


In [ ]:
LOGS = ROOT/'reports/r3/ast_platt_recipient_verification'/('software_'+uuid.uuid4().hex)
LOGS.mkdir(parents=True, exist_ok=False)
tests = subprocess.run([sys.executable,'-B','-m','unittest','discover','-s',str(VERIFY/'source/tests'),'-p','test_r3_n1_ast_platt.py','-v'],env=ENV,text=True,capture_output=True)
(LOGS/'software_tests.log').write_text(tests.stdout+tests.stderr)
print(tests.stdout, tests.stderr)
assert tests.returncode == 0 and 'Ran 7 tests' in tests.stderr and 'skipped=' not in tests.stderr, 'Stop: software verification failed'


In [ ]:
run = subprocess.run([sys.executable,'-B',str(VERIFY/'verify.py'),'--project-root',str(ROOT),'--candidate',str(CANDIDATE)],env=ENV,text=True,capture_output=True)
(LOGS/'waveform_verification.log').write_text(run.stdout+run.stderr)
print(run.stdout,run.stderr)
print('Durable logs:',LOGS)
assert run.returncode == 0, 'A discrepancy or missing dependency needs review. Preserve logs and report; do not retune or overwrite evidence.'


Return the newly created `reports/r3/ast_platt_recipient_verification/` reports and logs for review. Expected status is `sample_checks_passed_review_pending`, unchanged originals, matching sampled decisions and inherited raw-chunk parity. The reviewer must record candidate bundle/manifest identity, report identity, environment, findings and integration disposition. No cell promotes or rewrites a lock.
